# SIS #3 — The Warehouse Sequence Detective
## RNN vs. LSTM: Does the order of events matter?

**CSCI1409 · Deep Learning · Bachelor’s degree · Week 11 · 4 points**
**Format:** In-class team assignment, followed by an individual oral defense
**Team size:** 1–3 students · **Submission:** one completed `.ipynb` per team on Microsoft Teams

### Scenario
A warehouse camera system detects these five actions:

| Event ID | Action |
|:--:|---|
| `0` | IDLE |
| `1` | CHECK |
| `2` | MOVE |
| `3` | SCAN |
| `4` | PACK |

Your task is to classify a sequence of events:

- **Label `1` (correct):** `SCAN` occurred **before** `PACK`.
- **Label `0` (incorrect):** `PACK` occurred **before** `SCAN`.

Every sequence contains **exactly one SCAN and one PACK**, plus unrelated actions. So knowing *which* actions occurred is not enough: your model must learn **their order**.

**Example:** `IDLE → SCAN → MOVE → PACK` is class **1**, whereas `IDLE → PACK → MOVE → SCAN` is class **0**.

### What is provided
The **first code cell below is the entire provided dataset** and the only provided code. Run it once without changing it. It creates the train, validation and test sets in memory, using only NumPy. **No downloads, external files, data cleaning, or GPU are needed.**

The input arrays are `X_train`, `X_val`, `X_test` and the labels are `y_train`, `y_val`, `y_test`. Each input is a sequence of **16 one-hot event vectors** (5 features per time step).

### Your task — write ALL code yourselves
You must implement **every model, training loop, metric, and experiment from scratch**, using the **blank code cells** provided after the dataset. You may use standard PyTorch building blocks such as `nn.Linear`, `nn.RNN`, `nn.LSTM`, `CrossEntropyLoss`, `Adam` and `DataLoader`. You do **not** need to implement LSTM gates manually.

- **A. Explore + pipeline:** Inspect the data, then write reusable training and evaluation functions.
- **B. Baseline:** Train an order-blind model.
- **C. RNN:** Build and train a vanilla RNN classifier.
- **D. LSTM:** Build and train an LSTM classifier.
- **E. Test + reversal:** Evaluate all models on the test set once, then reverse the test sequences and compare predictions.
- **F. Conclusions:** Report your results and answer the short questions.
- **G. Optional challenges** (not graded).

Every model trains in a few seconds on a laptop CPU. Spend your time on **predicting** results before you run, **reading** your metrics, and **explaining** why each model behaves as it does: that is what the questions and the defense are about.

### Reproducibility
- Define `SEED = 0`. Call `torch.manual_seed(SEED)` **immediately before creating each model**, and seed the mini-batch shuffling, e.g. `DataLoader(..., shuffle=True, generator=torch.Generator().manual_seed(SEED))`.
- The numbers in your Part F table must be the ones your notebook prints after **Kernel → Restart & Run All**.

### Rules and AI policy


### Grading — 4.0 points

| Part | What you deliver | Points |
|---|---|---:|
| A | Data exploration; reusable training and evaluation functions | **0.50** |
| B | Order-blind baseline, trained and explained (F-Q1) | **0.50** |
| C | Vanilla RNN, trained and explained (F-Q2) | **0.75** |
| D | LSTM, trained and explained (F-Q3, F-Q4) | **0.75** |
| E | Test evaluation, reversal experiment and interpretation (F-Q5) | **0.50** |
| **Oral defense** | Individual understanding of the code and concepts | **1.00** |
| **Total** | | **4.00** |

Each part is graded on correct code, a reported result, and a correct explanation. Part G is optional and not graded.

**Suggested settings** (use the **same** settings for all models): `hidden_size=24`, `batch_size=128`, `epochs=10`, `learning_rate=0.01`, one recurrent layer. Use **2 output logits** for the two classes and cross-entropy loss.

---

### PROVIDED DATASET — run the following cell without editing it


In [1]:
import numpy as np

EVENTS = ["IDLE", "CHECK", "MOVE", "SCAN", "PACK"]
SEQUENCE_LENGTH = 16

def _make_split(n_samples, seed):
    rng = np.random.default_rng(seed)
    event_ids = rng.integers(0, 3, size=(n_samples, SEQUENCE_LENGTH), dtype=np.uint8)
    labels = np.array([0, 1] * (n_samples // 2), dtype=np.int64)
    rng.shuffle(labels)
    for i in range(n_samples):
        first, second = sorted(rng.choice(SEQUENCE_LENGTH, size=2, replace=False))
        event_ids[i, first], event_ids[i, second] = (
            (3, 4) if labels[i] == 1 else (4, 3)
        )
    return np.eye(5, dtype=np.float32)[event_ids], labels

X_train, y_train = _make_split(1200, seed=4)
X_val, y_val = _make_split(240, seed=5)
X_test, y_test = _make_split(240, seed=6)

print("Dataset ready — zero downloads required")
print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)


Dataset ready — zero downloads required
Train: (1200, 16, 5) (1200,)
Val:   (240, 16, 5) (240,)
Test:  (240, 16, 5) (240,)


## A — Explore the data and build the pipeline

### A1. Explore

**Write your own Python code** to:

1. Print the input shape and class counts for the train split.
2. Display **one example with label 1** and **one example with label 0** as readable event names, using the `EVENTS` list.
3. Inspect one time-step vector (for example, the first vector in the first sequence).

**Think about:** What do `N`, `T` and `5` represent in a shape `(N, T, 5)`? Why can two sequences containing the same number of SCAN and PACK events have different labels?

**Answer:** `N` is the number of sequences, `T` is the number of time steps, and `5` is the number of event features. The labels differ because the order of SCAN and PACK differs.


In [2]:
print("Train input shape:", X_train.shape)
print("Train class counts:", {int(label): int(count) for label, count in zip(*np.unique(y_train, return_counts=True))})

example_1 = int(np.where(y_train == 1)[0][0])
example_0 = int(np.where(y_train == 0)[0][0])

for label, index in [(1, example_1), (0, example_0)]:
    event_names = [EVENTS[event_id] for event_id in X_train[index].argmax(axis=1)]
    print(f"Label {label} example:", " → ".join(event_names))

print("First time-step vector:", X_train[0, 0])


Train input shape: (1200, 16, 5)
Train class counts: {0: 600, 1: 600}
Label 1 example: IDLE → IDLE → MOVE → MOVE → MOVE → IDLE → SCAN → MOVE → MOVE → IDLE → CHECK → MOVE → MOVE → PACK → MOVE → CHECK
Label 0 example: MOVE → CHECK → MOVE → MOVE → CHECK → IDLE → MOVE → PACK → IDLE → MOVE → CHECK → MOVE → SCAN → MOVE → MOVE → IDLE
First time-step vector: [1. 0. 0. 0. 0.]


### A2. Reusable training and evaluation functions

Using **PyTorch**, implement your own code that:

- Converts the provided NumPy arrays to suitable PyTorch tensors (which dtype does `CrossEntropyLoss` expect for the labels?).
- Trains a model on the **training split** using mini-batches, cross-entropy loss and Adam, with seeded shuffling.
- Performs the normal training steps: forward pass, loss, clear old gradients, backward pass and optimizer update.
- Computes **accuracy** without updating the model: use `model.eval()` and `torch.no_grad()`.
- Prints the training loss and the **validation** accuracy after every epoch.

*You will reuse these functions in Parts B–E.*

In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 0
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 0.01
HIDDEN_SIZE = 24

torch.set_num_threads(1)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

def accuracy(model, inputs, labels):
    model.eval()
    with torch.no_grad():
        predictions = model(inputs).argmax(dim=1)
        return (predictions == labels).float().mean().item()

def train_model(model):
    train_data = TensorDataset(X_train_t, y_train_t)
    loader = DataLoader(
        train_data,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        for inputs, labels in loader:
            optimizer.zero_grad()
            logits = model(inputs)
            loss = loss_function(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)

        average_loss = total_loss / len(train_data)
        val_accuracy = accuracy(model, X_val_t, y_val_t)
        print(f"Epoch {epoch + 1:2d}: loss={average_loss:.4f}, validation accuracy={val_accuracy:.3f}")

    return model


## B — Order-blind baseline

Create a small model that first **averages the 16 input vectors over the time dimension** and then applies a linear layer mapping **5 features → 2 class logits**.

Train it with your functions from Part A and report its **validation** accuracy. This model knows the **counts/proportions of events**, but not their order.

**Predict before you run:** can this model learn SCAN-before-PACK? Give a reason based on the *information lost by averaging*, then check your prediction.

**Prediction:** It cannot learn the order from the average. Every sequence has one SCAN and one PACK, so swapping their positions leaves the averaged features unchanged.


In [4]:
class BaselineClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(5, 2)

    def forward(self, inputs):
        average_events = inputs.mean(dim=1)
        return self.linear(average_events)

torch.manual_seed(SEED)
baseline_model = BaselineClassifier()
train_model(baseline_model)
baseline_val_accuracy = accuracy(baseline_model, X_val_t, y_val_t)
print(f"Baseline validation accuracy: {baseline_val_accuracy:.3f}")


Epoch  1: loss=0.6959, validation accuracy=0.483
Epoch  2: loss=0.6928, validation accuracy=0.483
Epoch  3: loss=0.6928, validation accuracy=0.483
Epoch  4: loss=0.6931, validation accuracy=0.488
Epoch  5: loss=0.6929, validation accuracy=0.508
Epoch  6: loss=0.6928, validation accuracy=0.508
Epoch  7: loss=0.6924, validation accuracy=0.483
Epoch  8: loss=0.6926, validation accuracy=0.492
Epoch  9: loss=0.6923, validation accuracy=0.479
Epoch 10: loss=0.6925, validation accuracy=0.463
Baseline validation accuracy: 0.463


## C — Vanilla RNN classifier

Implement a **new** PyTorch model using:

- A **single-layer `nn.RNN`** with `input_size=5`, `hidden_size=24`, `batch_first=True`.
- The **final hidden state** as the sequence representation.
- A linear classification head from **24 hidden features → 2 class logits**.

Train it with your functions from Part A and report its **validation** accuracy.

**Shape checkpoint:** With batch size `B`, the input is `(B, 16, 5)`; for a one-layer unidirectional RNN, the final hidden-state tensor is `(1, B, 24)`; the classifier output must be `(B, 2)`. **Print these three shapes for one small batch.**

**Defense checkpoint:** What information must the final hidden state carry for this task?

**Answer:** It must keep enough information about the events seen so far to tell whether SCAN came before PACK.


In [5]:
class RNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=5, hidden_size=HIDDEN_SIZE, batch_first=True)
        self.linear = nn.Linear(HIDDEN_SIZE, 2)

    def forward(self, inputs):
        outputs, h_n = self.rnn(inputs)
        return self.linear(h_n[-1])

torch.manual_seed(SEED)
rnn_model = RNNClassifier()
small_batch = X_train_t[:4]
with torch.no_grad():
    rnn_outputs, rnn_h_n = rnn_model.rnn(small_batch)
    rnn_logits = rnn_model.linear(rnn_h_n[-1])
print("Input shape:", tuple(small_batch.shape))
print("Final hidden-state shape:", tuple(rnn_h_n.shape))
print("Output logits shape:", tuple(rnn_logits.shape))

train_model(rnn_model)
rnn_val_accuracy = accuracy(rnn_model, X_val_t, y_val_t)
print(f"RNN validation accuracy: {rnn_val_accuracy:.3f}")


Input shape: (4, 16, 5)
Final hidden-state shape: (1, 4, 24)
Output logits shape: (4, 2)
Epoch  1: loss=0.6977, validation accuracy=0.579
Epoch  2: loss=0.6653, validation accuracy=0.729
Epoch  3: loss=0.5478, validation accuracy=0.858
Epoch  4: loss=0.2867, validation accuracy=0.896
Epoch  5: loss=0.1893, validation accuracy=0.938
Epoch  6: loss=0.0651, validation accuracy=0.967
Epoch  7: loss=0.1359, validation accuracy=0.967
Epoch  8: loss=0.0778, validation accuracy=0.975
Epoch  9: loss=0.0311, validation accuracy=0.958
Epoch 10: loss=0.0483, validation accuracy=0.983
RNN validation accuracy: 0.983


## D — LSTM classifier

Implement a **new** PyTorch model using:

- A **single-layer `nn.LSTM`** with `input_size=5`, `hidden_size=24`, `batch_first=True`.
- The **final hidden state `h_n`**, not the cell state `c_n`, as the input to the classifier.
- A linear classification head from **24 hidden features → 2 class logits**.

Train it with **exactly the same settings as the RNN** and report its **validation** accuracy.

**API checkpoint:** The RNN returns `(outputs, h_n)`, while the LSTM returns `(outputs, (h_n, c_n))`.

**Defense checkpoint:** In simple words, what is the purpose of `c_n`? How is it different from `h_n`? **You do not need to implement the gates from scratch.**

**Answer:** `c_n` is the LSTM's longer-term memory, updated as the sequence is read. `h_n` is the current hidden summary used to make the prediction.


In [6]:
class LSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=5, hidden_size=HIDDEN_SIZE, batch_first=True)
        self.linear = nn.Linear(HIDDEN_SIZE, 2)

    def forward(self, inputs):
        outputs, (h_n, c_n) = self.lstm(inputs)
        return self.linear(h_n[-1])

torch.manual_seed(SEED)
lstm_model = LSTMClassifier()
train_model(lstm_model)
lstm_val_accuracy = accuracy(lstm_model, X_val_t, y_val_t)
print(f"LSTM validation accuracy: {lstm_val_accuracy:.3f}")


Epoch  1: loss=0.6882, validation accuracy=0.654
Epoch  2: loss=0.6312, validation accuracy=0.833
Epoch  3: loss=0.3354, validation accuracy=0.917
Epoch  4: loss=0.1293, validation accuracy=0.933
Epoch  5: loss=0.0689, validation accuracy=0.983
Epoch  6: loss=0.0376, validation accuracy=0.992
Epoch  7: loss=0.0164, validation accuracy=0.992
Epoch  8: loss=0.0092, validation accuracy=0.996
Epoch  9: loss=0.0051, validation accuracy=0.996
Epoch 10: loss=0.0048, validation accuracy=0.988
LSTM validation accuracy: 0.988


## E — Final test evaluation and the reversal experiment

All three models are now trained. Do not change any model or setting from here on.

1. Evaluate the baseline, RNN and LSTM on the **test set, once**. These are your reported test accuracies.
2. Reverse each test sequence **along the time dimension only** (keep its 5 features intact).
3. Check that the number of each type of event is unchanged by reversal.
4. Evaluate all three models on the reversed inputs **against the original `y_test` labels**. Label this as a **diagnostic**, not normal test accuracy.
5. Compare predictions: compute the **flip rate**, the fraction of test sequences whose predicted class changes after reversal.
6. Reason about the **correct** labels for the reversed sequences. (Hint: every sequence contains exactly one SCAN and one PACK.)

**Before you conclude anything:** a model can score close to 0% in step 4. Think about what that number means before you decide whether the model is broken.

**Questions:** Which models respond when the order changes? Why might the order-blind model give the same predictions after reversal?

**Answer:** Reversing a sequence swaps which event came first, so its correct label changes from `y_test` to `1 - y_test`. A low reversed diagnostic accuracy against the old labels can mean the model learned the order correctly. The baseline averages over time, so reversal leaves its input and predictions unchanged.


In [7]:
def predict(model, inputs):
    model.eval()
    with torch.no_grad():
        return model(inputs).argmax(dim=1)

models = {
    "Order-blind baseline": baseline_model,
    "Vanilla RNN": rnn_model,
    "LSTM": lstm_model,
}

# Get the original test predictions once, then use them for accuracy and flip rate.
original_predictions = {
    name: predict(model, X_test_t)
    for name, model in models.items()
}
test_accuracies = {
    name: (original_predictions[name] == y_test_t).float().mean().item()
    for name in models
}

X_test_reversed = torch.flip(X_test_t, dims=[1])
event_counts_same = torch.equal(X_test_t.sum(dim=1), X_test_reversed.sum(dim=1))
print("Event counts unchanged by reversal:", event_counts_same)
print("Correct labels after reversal are 1 - y_test.")

reversed_accuracies = {}
flip_rates = {}
for name, model in models.items():
    reversed_predictions = predict(model, X_test_reversed)
    reversed_accuracies[name] = (reversed_predictions == y_test_t).float().mean().item()
    flip_rates[name] = (reversed_predictions != original_predictions[name]).float().mean().item()
    print(
        f"{name}: test accuracy={test_accuracies[name]:.3f}, "
        f"reversed diagnostic={reversed_accuracies[name]:.3f}, "
        f"flip rate={flip_rates[name]:.3f}"
    )


Event counts unchanged by reversal: True
Correct labels after reversal are 1 - y_test.
Order-blind baseline: test accuracy=0.512, reversed diagnostic=0.512, flip rate=0.000
Vanilla RNN: test accuracy=0.992, reversed diagnostic=0.017, flip rate=0.975
LSTM: test accuracy=1.000, reversed diagnostic=0.004, flip rate=0.996


## F — Results and conclusions

### Results table

| Model | Validation accuracy | Test accuracy | Reversed vs. original labels (diagnostic) | Flip rate |
|---|---:|---:|---:|---:|
| Order-blind baseline | 0.463 | 0.512 | 0.512 | 0.000 |
| Vanilla RNN | 0.983 | 0.992 | 0.017 | 0.975 |
| LSTM | 0.988 | 1.000 | 0.004 | 0.996 |

### Short answers (1–3 sentences each)

**1.** Why can a model that averages events over time not reliably distinguish SCAN-before-PACK from PACK-before-SCAN?

**Answer:** Averaging keeps the event counts but removes their positions. Since every sequence has one SCAN and one PACK, both orders have the same average.

**2.** In `(B, T, 5)`, what are `B` and `T`? What is the shape of the final output logits for `B = 8`? Why?

**Answer:** `B` is the number of sequences in the batch, and `T` is the number of time steps in each sequence. The logits have shape `(8, 2)`: one pair of class scores for each sequence.

**3.** What is the difference between the hidden state `h_n` and the LSTM cell state `c_n`?

**Answer:** `c_n` carries the LSTM's longer-term memory through the sequence. `h_n` is the current hidden summary, which this model sends to the classifier.

**4.** Which model worked best **in your experiment**? Is the difference between the RNN and the LSTM large enough to matter here? Does this mean LSTM always beats a vanilla RNN? Why or why not?

**Answer:** The LSTM was slightly best: its validation accuracy was 0.988 versus 0.983 for the RNN, and its test accuracy was 1.000 versus 0.992. These small gaps do not show that an LSTM always wins; results depend on the task, data, settings, and random seed.

**5.** After reversal, an order-aware model can score close to 0% against the **original** labels. Is that a failure? Explain what this diagnostic and the flip rate tell you about which models use order.

**Answer:** No. Reversal changes the correct label to `1 - y_test`, so a model that learned order can be right for the reversed sequence and wrong against the original label. The RNN and LSTM flip predictions for most sequences, while the baseline's predictions stay the same because averaging is unchanged.


## G — Optional challenges (not graded)

Only if you have finished A–F. None of these affects your 4 points, but they make good defense material.

- **G1. Is the difference real?** Retrain the RNN and LSTM with seeds `0, 1, 2` and compare their **validation** accuracies. Is the RNN–LSTM gap larger than the difference between two seeds of the same model?
- **G2. Confusion matrices.** Write your own 2×2 confusion matrix (rows = true class, columns = predicted class) and print it for each model on the original and the reversed test inputs. What pattern appears for the RNN after reversal, and why does the baseline’s matrix not change?
- **G3. Parameters.** Derive by hand the number of parameters of your RNN and LSTM, check with `sum(p.numel() for p in model.parameters())`, and explain why the LSTM has about 4× more.
- **G4. Longer sequences (no code).** In Homework 7 you computed how an early input’s influence shrinks as $w^{T-1}$. Using this, explain why a vanilla RNN would struggle if SCAN and PACK could be 50 steps before the end. How does the LSTM update $c_t = f \cdot c_{t-1} + i \cdot g$ avoid this when the forget gate $f$ is close to 1?

*Use the empty cell below if you attempt G1–G3, and write G4 as text.*

### Team members and contributions

- **Student 1** — Ibrayev Miras / 24B031810 / Code, experiments, and written answers
- **Student 2** — [Full name / student ID / contribution, if applicable]
- **Student 3** — [Full name / student ID / contribution, if applicable]

### Before submitting

- **Kernel → Restart & Run All** finishes without errors, and the numbers in your Part F table match the printed outputs.
- The test set is evaluated only in Part E.
- Each team member can explain any chosen code line during the **individual oral defense**.

**Submission:** One completed `.ipynb` per team via Microsoft Teams. No separate report required.
